# Session 5 — Time Series & I/O

Runnable code for the **Build It** and **Experiment** sections.
Run each cell top to bottom.

In [1]:
import io
import os

import numpy as np
import pandas as pd

## 1. Build It — Core Code

### 5.1 Parsing dates with `pd.to_datetime`

In [2]:
raw = pd.Series(["2024-01-01", "2024-02-15", "2024-03-20"])
parsed = pd.to_datetime(raw)
print(parsed.dtype)
parsed

datetime64[us]


0   2024-01-01
1   2024-02-15
2   2024-03-20
dtype: datetime64[us]

In [3]:
# Unparseable values become NaT instead of raising, thanks to errors="coerce"
pd.to_datetime(["2024-01-01", "not-a-date"], errors="coerce")

DatetimeIndex(['2024-01-01', 'NaT'], dtype='datetime64[us]', freq=None)

### 5.2 The `.dt` accessor

In [4]:
daily = pd.DataFrame({
    "date": pd.date_range("2024-01-01", periods=14, freq="D"),
    "sales": [220, 180, 310, 260, 90, 150, 400,
              210, 175, 330, 275, 100, 160, 420],
})
daily["date"] = pd.to_datetime(daily["date"])
daily["day_name"] = daily["date"].dt.day_name()
daily["month"] = daily["date"].dt.month
daily["weekday_num"] = daily["date"].dt.dayofweek
daily.head()

,date,sales,day_name,month,weekday_num
0,2024-01-01,220,Monday,1,0
1,2024-01-02,180,Tuesday,1,1
2,2024-01-03,310,Wednesday,1,2
3,2024-01-04,260,Thursday,1,3
4,2024-01-05,90,Friday,1,4


In [5]:
daily["day_name"].value_counts()

day_name
Monday       2
Tuesday      2
Wednesday    2
Thursday     2
Friday       2
Saturday     2
Sunday       2
Name: count, dtype: int64

### 5.3 Setting a `DatetimeIndex`

In [6]:
ts = daily.set_index("date").sort_index()
ts.head()

,sales,day_name,month,weekday_num
date,,,,
2024-01-01,220,Monday,1,0
2024-01-02,180,Tuesday,1,1
2024-01-03,310,Wednesday,1,2
2024-01-04,260,Thursday,1,3
2024-01-05,90,Friday,1,4


In [7]:
ts.loc["2024-01-05":"2024-01-08", "sales"]

date
2024-01-05     90
2024-01-06    150
2024-01-07    400
2024-01-08    210
Name: sales, dtype: int64

### 5.4 `resample` — groupby for time

In [8]:
ts["sales"].resample("W").sum()

date
2024-01-07    1610
2024-01-14    1670
Freq: W-SUN, Name: sales, dtype: int64

In [9]:
ts["sales"].resample("W").agg(["sum", "mean", "max"])

,sum,mean,max
date,,,
2024-01-07,1610,230.000000,400
2024-01-14,1670,238.571429,420


### 5.5 `rolling` — moving windows

In [10]:
ts["sales"].rolling(window=3).mean()

date
2024-01-01           NaN
2024-01-02           NaN
2024-01-03    236.666667
2024-01-04    250.000000
2024-01-05    220.000000
2024-01-06    166.666667
2024-01-07    213.333333
2024-01-08    253.333333
2024-01-09    261.666667
2024-01-10    238.333333
2024-01-11    260.000000
2024-01-12    235.000000
2024-01-13    178.333333
2024-01-14    226.666667
Name: sales, dtype: float64

In [11]:
ts["sales"].rolling(window=3, min_periods=1).mean()

date
2024-01-01    220.000000
2024-01-02    200.000000
2024-01-03    236.666667
2024-01-04    250.000000
2024-01-05    220.000000
2024-01-06    166.666667
2024-01-07    213.333333
2024-01-08    253.333333
2024-01-09    261.666667
2024-01-10    238.333333
2024-01-11    260.000000
2024-01-12    235.000000
2024-01-13    178.333333
2024-01-14    226.666667
Name: sales, dtype: float64

### 5.6 `shift`, `diff`, `pct_change`

In [12]:
changes = pd.DataFrame({
    "sales": ts["sales"],
    "lag_1": ts["sales"].shift(1),
    "diff": ts["sales"].diff(),
    "pct_change": ts["sales"].pct_change().round(4),
})
changes

,sales,lag_1,diff,pct_change
date,,,,
2024-01-01,220,NaN,NaN,NaN
2024-01-02,180,220.0,-40.0,-0.1818
2024-01-03,310,180.0,130.0,0.7222
2024-01-04,260,310.0,-50.0,-0.1613
2024-01-05,90,260.0,-170.0,-0.6538
2024-01-06,150,90.0,60.0,0.6667
2024-01-07,400,150.0,250.0,1.6667
2024-01-08,210,400.0,-190.0,-0.4750
2024-01-09,175,210.0,-35.0,-0.1667


### 5.7 CSV — `to_csv` / `read_csv`

We use `io.StringIO` so nothing touches the disk.

In [13]:
csv_text = ts.reset_index().to_csv(index=False)
csv_back = pd.read_csv(io.StringIO(csv_text), parse_dates=["date"])
print(csv_back.dtypes)
csv_back.head()

date           datetime64[us]
sales                   int64
day_name                  str
month                   int64
weekday_num             int64
dtype: object


,date,sales,day_name,month,weekday_num
0,2024-01-01,220,Monday,1,0
1,2024-01-02,180,Tuesday,1,1
2,2024-01-03,310,Wednesday,1,2
3,2024-01-04,260,Thursday,1,3
4,2024-01-05,90,Friday,1,4


### 5.8 Excel — `to_excel` / `read_excel`

In [14]:
xlsx_path = "demo_sales.xlsx"
ts.reset_index().to_excel(xlsx_path, index=False, sheet_name="Sales")
excel_back = pd.read_excel(xlsx_path, sheet_name="Sales", parse_dates=["date"])
print(excel_back.dtypes)
os.remove(xlsx_path)
excel_back.head()

date           datetime64[us]
sales                   int64
day_name                  str
month                   int64
weekday_num             int64
dtype: object


,date,sales,day_name,month,weekday_num
0,2024-01-01,220,Monday,1,0
1,2024-01-02,180,Tuesday,1,1
2,2024-01-03,310,Wednesday,1,2
3,2024-01-04,260,Thursday,1,3
4,2024-01-05,90,Friday,1,4


### 5.9 Parquet — `to_parquet` / `read_parquet`

In [15]:
parquet_path = "demo_sales.parquet"
ts.reset_index().to_parquet(parquet_path, index=False)
parquet_back = pd.read_parquet(parquet_path)
print(parquet_back.dtypes)
os.remove(parquet_path)
parquet_back.head()

date           datetime64[us]
sales                   int64
day_name                  str
month                   int32
weekday_num             int32
dtype: object


,date,sales,day_name,month,weekday_num
0,2024-01-01,220,Monday,1,0
1,2024-01-02,180,Tuesday,1,1
2,2024-01-03,310,Wednesday,1,2
3,2024-01-04,260,Thursday,1,3
4,2024-01-05,90,Friday,1,4


## 2. Experiment

Change a parameter, predict the output, then run the cell.

In [16]:
# Experiment 1: daily vs weekly vs monthly resampling of the same series
print("Daily sum:")
print(ts["sales"].resample("D").sum().head(3))
print("\nWeekly sum:")
print(ts["sales"].resample("W").sum().head(3))
print("\nMonthly sum:")
print(ts["sales"].resample("ME").sum())

Daily sum:
date
2024-01-01    220
2024-01-02    180
2024-01-03    310
Freq: D, Name: sales, dtype: int64

Weekly sum:
date
2024-01-07    1610
2024-01-14    1670
Freq: W-SUN, Name: sales, dtype: int64

Monthly sum:
date
2024-01-31    3280
Freq: ME, Name: sales, dtype: int64


In [17]:
# Experiment 2: bigger windows smooth more but add more leading NaNs
for w in (2, 3, 5):
    nan_count = ts["sales"].rolling(window=w).mean().isna().sum()
    print(f"window={w}: leading NaNs = {nan_count}")

window=2: leading NaNs = 1
window=3: leading NaNs = 2
window=5: leading NaNs = 4


In [18]:
# Experiment 3: shift periods move the series in time
pd.DataFrame({
    "sales": ts["sales"],
    "shift_1": ts["sales"].shift(1),
    "shift_-1": ts["sales"].shift(-1),
}).head()

,sales,shift_1,shift_-1
date,,,
2024-01-01,220,NaN,180.0
2024-01-02,180,220.0,310.0
2024-01-03,310,180.0,260.0
2024-01-04,260,310.0,90.0
2024-01-05,90,260.0,150.0


In [19]:
# Experiment 4: diff vs pct_change on the same rows
pd.DataFrame({
    "sales": ts["sales"],
    "diff": ts["sales"].diff(),
    "pct_change": ts["sales"].pct_change().round(4),
}).head()

,sales,diff,pct_change
date,,,
2024-01-01,220,NaN,NaN
2024-01-02,180,-40.0,-0.1818
2024-01-03,310,130.0,0.7222
2024-01-04,260,-50.0,-0.1613
2024-01-05,90,-170.0,-0.6538


In [20]:
# Experiment 5: read_csv without parse_dates leaves the date as text
no_parse = pd.read_csv(io.StringIO(csv_text))
with_parse = pd.read_csv(io.StringIO(csv_text), parse_dates=["date"])
print("without parse_dates:", no_parse["date"].dtype)
print("with parse_dates:   ", with_parse["date"].dtype)

without parse_dates: str
with parse_dates:    datetime64[us]


## 3. Mini Project — Daily Sales Time Series

Generate a year of daily sales, resample it, smooth it with a rolling mean,
measure growth, and save it in three formats.

In [21]:
# Step 1: generate a year of daily sales with trend + weekly seasonality + noise
rng = np.random.default_rng(7)
dates = pd.date_range("2024-01-01", periods=365, freq="D")
trend = np.linspace(200, 400, 365)
weekly = 60 * np.sin(np.arange(365) * 2 * np.pi / 7)
noise = rng.normal(0, 25, 365)

sales = pd.DataFrame({"date": dates, "sales": (trend + weekly + noise).round(0)})
sales.head()

,date,sales
0,2024-01-01,200.0
1,2024-01-02,255.0
2,2024-01-03,253.0
3,2024-01-04,205.0
4,2024-01-05,165.0


In [22]:
# Step 2: index by date and compute the monthly totals
ts = sales.set_index("date").sort_index()
monthly = ts["sales"].resample("ME").sum()
monthly

date
2024-01-31     6213.0
2024-02-29     6575.0
2024-03-31     7295.0
2024-04-30     7714.0
2024-05-31     8396.0
2024-06-30     8435.0
2024-07-31     9833.0
2024-08-31     9785.0
2024-09-30    10083.0
2024-10-31    11303.0
2024-11-30    11148.0
2024-12-31    11823.0
Freq: ME, Name: sales, dtype: float64

In [23]:
# Step 3: add a 7-day rolling mean and day-over-day growth
ts["rolling_7"] = ts["sales"].rolling(7).mean()
ts["pct_change"] = ts["sales"].pct_change().round(4)
ts[["sales", "rolling_7", "pct_change"]].head(10)

,sales,rolling_7,pct_change
date,,,
2024-01-01,200.0,NaN,NaN
2024-01-02,255.0,NaN,0.2750
2024-01-03,253.0,NaN,-0.0078
2024-01-04,205.0,NaN,-0.1897
2024-01-05,165.0,NaN,-0.1951
2024-01-06,119.0,NaN,-0.2788
2024-01-07,158.0,193.571429,0.3277
2024-01-08,237.0,198.857143,0.5000
2024-01-09,239.0,196.571429,0.0084


In [24]:
# Step 4: best and worst sales days
print("Best day: ", ts["sales"].idxmax(), ts["sales"].max())
print("Worst day:", ts["sales"].idxmin(), ts["sales"].min())

Best day:  2024-12-04 00:00:00 471.0
Worst day: 2024-01-27 00:00:00 93.0


In [25]:
# Step 5: save to three formats, reload, confirm row counts, then clean up
csv_path, xlsx_path, parquet_path = "mini.csv", "mini.xlsx", "mini.parquet"
out = ts.reset_index()
out.to_csv(csv_path, index=False)
out.to_excel(xlsx_path, index=False, sheet_name="Sales")
out.to_parquet(parquet_path, index=False)

csv_back = pd.read_csv(csv_path, parse_dates=["date"])
xlsx_back = pd.read_excel(xlsx_path, sheet_name="Sales", parse_dates=["date"])
parquet_back = pd.read_parquet(parquet_path)

print("CSV rows:    ", len(csv_back))
print("Excel rows:  ", len(xlsx_back))
print("Parquet rows:", len(parquet_back))

os.remove(csv_path)
os.remove(xlsx_path)
os.remove(parquet_path)

CSV rows:     365
Excel rows:   365
Parquet rows: 365
